In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import os

import numpy as np
import pandas as pd

from pymongo import MongoClient, ASCENDING
from dotenv import load_dotenv

load_dotenv()

False

In [2]:
PROMO_DIR = Path("data/external/walmart/promotions")
RAW_DIR = PROMO_DIR / "raw"
PROCESSED_DIR = PROMO_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = RAW_DIR / "walmart_on_flyer_2026-09-10.csv"

OUTPUT_PATH = (
    PROCESSED_DIR /
    "walmart_on_promotions.parquet"
)

In [3]:
columns = [
    "flyer_week_start",
    "flyer_week_end",
    "region",
    "store_id",
    "product_name",
    "brand",
    "category",
    "promo_price",
    "regular_price",
    "unit_price_text",
    "promotion_type",
    "availability_text",
    "source_url",
    "captured_at"
]

if not INPUT_PATH.exists():

    pd.DataFrame(columns=columns).to_csv(
        INPUT_PATH,
        index=False
    )

    print("Created:", INPUT_PATH)

else:
    print("Already exists:", INPUT_PATH)

Created: data/external/walmart/promotions/raw/walmart_on_flyer_2026-09-10.csv


# Load

In [4]:
promos_raw = pd.read_csv(INPUT_PATH)

print(promos_raw.shape)

display(promos_raw.head())

(0, 14)


,flyer_week_start,flyer_week_end,region,store_id,product_name,brand,category,promo_price,regular_price,unit_price_text,promotion_type,availability_text,source_url,captured_at


In [5]:
promos_raw["flyer_week_start"] = pd.to_datetime(
    promos_raw["flyer_week_start"],
    errors="coerce"
)

promos_raw["flyer_week_end"] = pd.to_datetime(
    promos_raw["flyer_week_end"],
    errors="coerce"
)

promos_raw["captured_at"] = pd.to_datetime(
    promos_raw["captured_at"],
    errors="coerce",
    utc=True
)

In [6]:
now = pd.Timestamp.now(tz="UTC")

promos_raw["captured_at"] = (
    promos_raw["captured_at"]
    .fillna(now)
)

# Price cleanup

In [7]:
for col in [
    "promo_price",
    "regular_price"
]:

    promos_raw[col] = pd.to_numeric(
        promos_raw[col],
        errors="coerce"
    )

# Promotion features

In [8]:
promos = promos_raw.copy()

promos["discount_amount"] = (
    promos["regular_price"]
    - promos["promo_price"]
)

promos["discount_pct"] = np.where(
    promos["regular_price"] > 0,

    (
        promos["discount_amount"]
        / promos["regular_price"]
    ) * 100,

    np.nan
)

In [9]:
promos["is_promoted"] = (
    (
        promos["promo_price"].notna()
        & promos["regular_price"].notna()
        & (
            promos["promo_price"]
            < promos["regular_price"]
        )
    )
    |
    promos["promotion_type"].notna()
).astype("int8")

# Basic validation

In [10]:
print(
    "Rows:",
    len(promos)
)

print(
    "Promoted:",
    promos["is_promoted"].sum()
)

print(
    "Missing product:",
    promos["product_name"].isna().sum()
)

print(
    "Bad prices:",
    (
        promos["promo_price"] < 0
    ).sum()
)

Rows: 0
Promoted: 0
Missing product: 0
Bad prices: 0


In [11]:
display(
    promos[
        [
            "product_name",
            "regular_price",
            "promo_price",
            "discount_amount",
            "discount_pct",
            "promotion_type"
        ]
    ].head(20)
)

,product_name,regular_price,promo_price,discount_amount,discount_pct,promotion_type


# Create deterministic promotion ID

In [12]:
def make_promo_id(row):

    text = "|".join([
        str(row.get("region", "")),
        str(row.get("store_id", "")),
        str(row.get("flyer_week_start", "")),
        str(row.get("product_name", "")).lower().strip()
    ])

    return hashlib.sha1(
        text.encode("utf-8")
    ).hexdigest()[:20]

In [13]:
promos["promo_id"] = promos.apply(
    make_promo_id,
    axis=1
)

In [14]:
print(
    "Rows:",
    len(promos)
)

print(
    "Unique promo IDs:",
    promos["promo_id"].nunique()
)

Rows: 0
Unique promo IDs: 0


# MongoDB

In [15]:
# Connect

MONGO_URI = os.getenv(
    "MONGODB_URI",
    "mongodb://localhost:27017/"
)

client = MongoClient(MONGO_URI)

db = client[
    "retail_promotion_intelligence"
]

raw_collection = db[
    "flyer_snapshots"
]

promo_collection = db[
    "promotions"
]

client.admin.command("ping")

print("MongoDB connected")

MongoDB connected


# Create indexes

In [16]:
promo_collection.create_index(
    [("promo_id", ASCENDING)],
    unique=True
)

promo_collection.create_index(
    [
        ("region", ASCENDING),
        ("flyer_week_start", ASCENDING)
    ]
)

promo_collection.create_index(
    [("product_name", ASCENDING)]
)

'product_name_1'

# Store raw flyer snapshot

In [17]:
raw_records = promos_raw.copy()

In [18]:
for col in [
    "flyer_week_start",
    "flyer_week_end",
    "captured_at"
]:

    raw_records[col] = raw_records[col].apply(
        lambda x:
            x.to_pydatetime()
            if pd.notna(x)
            else None
    )

In [19]:
records = (
    raw_records
    .astype(object)
    .where(
        pd.notna(raw_records),
        None
    )
    .to_dict("records")
)

In [20]:
if records:

    raw_collection.insert_many(
        records
    )

print(
    "Raw snapshots:",
    raw_collection.count_documents({})
)

Raw snapshots: 0


# Store normalized promotion documents

In [21]:
mongo_promos = promos.copy()

for col in [
    "flyer_week_start",
    "flyer_week_end",
    "captured_at"
]:

    mongo_promos[col] = mongo_promos[col].apply(
        lambda x:
            x.to_pydatetime()
            if pd.notna(x)
            else None
    )

In [22]:
promo_records = (
    mongo_promos
    .astype(object)
    .where(
        pd.notna(mongo_promos),
        None
    )
    .to_dict("records")
)

In [23]:
for record in promo_records:

    promo_collection.update_one(
        {
            "promo_id":
                record["promo_id"]
        },
        {
            "$set": record
        },
        upsert=True
    )

In [24]:
print(
    "Normalized promotions:",
    promo_collection.count_documents({})
)

Normalized promotions: 0


# Save modeling table

In [26]:
model_cols = [
    "promo_id",
    "flyer_week_start",
    "flyer_week_end",
    "region",
    "store_id",
    "product_name",
    "brand",
    "category",
    "regular_price",
    "promo_price",
    "discount_amount",
    "discount_pct",
    "promotion_type",
    "is_promoted"
]

promotion_table = (
    promos[model_cols]
    .drop_duplicates(
        subset=["promo_id"]
    )
    .copy()
)

In [27]:
promotion_table.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)

Saved: data/external/walmart/promotions/processed/walmart_on_promotions.parquet


# Final QA

In [28]:
print(
    "Promotions:",
    len(promotion_table)
)

print(
    "Products:",
    promotion_table[
        "product_name"
    ].nunique()
)

print(
    "Categories:",
    promotion_table[
        "category"
    ].nunique()
)

print(
    "Weeks:",
    promotion_table[
        "flyer_week_start"
    ].nunique()
)

print(
    "Median discount:",
    promotion_table[
        "discount_pct"
    ].median()
)

Promotions: 0
Products: 0
Categories: 0
Weeks: 0
Median discount: nan


In [29]:
display(
    promotion_table
    .sort_values(
        "discount_pct",
        ascending=False
    )
    .head(20)
)

,promo_id,flyer_week_start,flyer_week_end,region,store_id,product_name,brand,category,regular_price,promo_price,discount_amount,discount_pct,promotion_type,is_promoted


# Verify

In [30]:
# 1. Verify source CSV
print("CSV rows:", len(promos_raw))
print("Missing product names:", promos_raw["product_name"].isna().sum())
print("Missing promo prices:", promos_raw["promo_price"].isna().sum())

display(promos_raw.head())

CSV rows: 0
Missing product names: 0
Missing promo prices: 0


,flyer_week_start,flyer_week_end,region,store_id,product_name,brand,category,promo_price,regular_price,unit_price_text,promotion_type,availability_text,source_url,captured_at


In [31]:
# 2. Verify promotion calculations
display(
    promos[
        [
            "product_name",
            "regular_price",
            "promo_price",
            "discount_amount",
            "discount_pct",
            "is_promoted"
        ]
    ].head(20)
)

print("Promoted rows:", promos["is_promoted"].sum())
print("Duplicate promo IDs:", promos["promo_id"].duplicated().sum())

,product_name,regular_price,promo_price,discount_amount,discount_pct,is_promoted


Promoted rows: 0
Duplicate promo IDs: 0


In [32]:
# 3. Verify MongoDB
print(
    "Raw flyer snapshots:",
    raw_collection.count_documents({})
)

print(
    "Normalized promotions:",
    promo_collection.count_documents({})
)

Raw flyer snapshots: 0
Normalized promotions: 0


In [33]:
docs = list(
    promo_collection.find(
        {},
        {"_id": 0}
    ).limit(5)
)

display(pd.DataFrame(docs))

""


In [34]:
# 4. Verify Parquet
check = pd.read_parquet(
    "data/external/walmart/promotions/processed/"
    "walmart_on_promotions.parquet"
)

print("Shape:", check.shape)
print("Columns:", check.columns.tolist())

display(check.head())

Shape: (0, 14)
Columns: ['promo_id', 'flyer_week_start', 'flyer_week_end', 'region', 'store_id', 'product_name', 'brand', 'category', 'regular_price', 'promo_price', 'discount_amount', 'discount_pct', 'promotion_type', 'is_promoted']


,promo_id,flyer_week_start,flyer_week_end,region,store_id,product_name,brand,category,regular_price,promo_price,discount_amount,discount_pct,promotion_type,is_promoted


In [35]:
print(
    "Duplicate promo IDs:",
    check["promo_id"].duplicated().sum()
)

print(
    "Negative promo prices:",
    (check["promo_price"] < 0).sum()
)

print(
    "Discount > 100%:",
    (check["discount_pct"] > 100).sum()
)

print(
    "Missing product names:",
    check["product_name"].isna().sum()
)

Duplicate promo IDs: 0
Negative promo prices: 0
Discount > 100%: 0
Missing product names: 0


In [36]:
print("CSV:", len(promos_raw))
print("Normalized dataframe:", len(promos))
print("Parquet:", len(check))
print("Mongo promotions:", promo_collection.count_documents({}))

CSV: 0
Normalized dataframe: 0
Parquet: 0
Mongo promotions: 0
